# Epi Info AI conditional LOGISTIC validation lab — V0.1

Independently validate the bounded browser candidate for `LOGISTIC outcome = predictors MATCHVAR=matched_set`. This notebook uses SciPy and the raw checked-in data; it does not call the TypeScript implementation under test.

## Candidate boundary

V0.1 requires one case and at least one control per matched set, complete binary outcome values, and simple numeric or binary predictors. The set-specific intercepts are removed by conditioning. Ordinary logistic regression, categorical expansion, interactions, weights, `OUTTABLE`, and desktop-parity claims remain out of scope.

In [ ]:
import csv, hashlib, io, math, sys
from collections import defaultdict
import numpy as np
import scipy
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import chi2, norm
from pyodide.http import pyfetch

fixture_response = await pyfetch('../../validation-fixtures/conditional-logistic-v0.1.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
data_response = await pyfetch('../../examples/matched-case-control/matched-logistic-test-data.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
normalized_bytes = data_bytes.replace(b'\r\n', b'\n')
assert hashlib.sha256(normalized_bytes).hexdigest() == fixture['stressDataset']['normalizedSha256']
all_records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
iteration = str(fixture['stressDataset']['iterationValue'])
records = [record for record in all_records if record[fixture['stressDataset']['iterationField']] == iteration]
assert len(records) == fixture['stressDataset']['records'] == 300
{'python': sys.version, 'scipy': scipy.__version__, 'records': len(records), 'candidate': fixture['candidateVersion']}

In [ ]:
paired = fixture['pairedIdentity']
b = paired['discordantCaseExposed']; c = paired['discordantControlExposed']
paired_independent = {
    'coefficient': math.log(b / c),
    'standardError': math.sqrt(1 / b + 1 / c),
    'oddsRatio': b / c,
}
for name, value in paired_independent.items():
    assert math.isclose(value, paired[name], rel_tol=0, abs_tol=1e-12), (name, value, paired[name])
paired_independent

In [ ]:
def derive_sets(source, match_field, outcome_field, predictor_fields):
    groups = defaultdict(list)
    for record in source:
        groups[record[match_field]].append(record)
    result = []
    for match_id, members in groups.items():
        outcomes = np.array([int(member[outcome_field]) for member in members], dtype=int)
        design = np.array([[float(member[field]) for field in predictor_fields] for member in members], dtype=float)
        assert outcomes.sum() == 1 and len(members) >= 2, match_id
        result.append((design, int(np.flatnonzero(outcomes == 1)[0])))
    return result

def fit_conditional(source):
    spec = fixture['stressDataset']; predictor_fields = spec['predictorFields']
    sets = derive_sets(source, spec['matchField'], spec['outcomeField'], predictor_fields)
    def objective(beta):
        return sum(logsumexp(design @ beta) - design[case_index] @ beta for design, case_index in sets)
    def gradient(beta):
        total = np.zeros(len(beta))
        for design, case_index in sets:
            eta = design @ beta; probabilities = np.exp(eta - logsumexp(eta))
            total += probabilities @ design - design[case_index]
        return total
    def information(beta):
        total = np.zeros((len(beta), len(beta)))
        for design, _ in sets:
            eta = design @ beta; probabilities = np.exp(eta - logsumexp(eta))
            mean = probabilities @ design; centered = design - mean
            total += (centered * probabilities[:, None]).T @ centered
        return total
    zero = np.zeros(len(predictor_fields))
    fitted = minimize(objective, zero, jac=gradient, hess=information, method='trust-exact', options={'gtol': 1e-10, 'maxiter': 100})
    assert fitted.success or np.linalg.norm(gradient(fitted.x), ord=np.inf) < 1e-8, fitted.message
    covariance = np.linalg.inv(information(fitted.x)); standard_errors = np.sqrt(np.diag(covariance))
    z = fitted.x / standard_errors; odds_ratios = np.exp(fitted.x)
    log_likelihood = -float(fitted.fun); null_log_likelihood = -float(objective(zero))
    likelihood_ratio = 2 * (log_likelihood - null_log_likelihood)
    return {
        'sets': len(sets), 'records': sum(len(design) for design, _ in sets),
        'coefficients': fitted.x, 'standardErrors': standard_errors, 'oddsRatios': odds_ratios,
        'z': z, 'pValues': 2 * norm.sf(np.abs(z)),
        'lower95': np.exp(fitted.x - norm.ppf(.975) * standard_errors),
        'upper95': np.exp(fitted.x + norm.ppf(.975) * standard_errors),
        'logLikelihood': log_likelihood, 'nullLogLikelihood': null_log_likelihood,
        'likelihoodRatio': likelihood_ratio, 'modelPValue': chi2.sf(likelihood_ratio, len(fitted.x)),
    }

independent = fit_conditional(records)
independent

In [ ]:
expected = fixture['candidateExpected']; tolerance = fixture['tolerance']
assert independent['sets'] == expected['totals']['includedSets'] == 100
assert independent['records'] == expected['totals']['includedRecords'] == 300
for index, candidate in enumerate(expected['coefficients']):
    comparisons = {
        'coefficient': independent['coefficients'][index],
        'standardError': independent['standardErrors'][index],
        'oddsRatio': independent['oddsRatios'][index],
        'lower95': independent['lower95'][index],
        'upper95': independent['upper95'][index],
    }
    for name, actual in comparisons.items():
        limit = tolerance.get(name, tolerance['oddsRatio'])
        assert math.isclose(float(actual), candidate[name], rel_tol=0, abs_tol=limit), (candidate['field'], name, actual, candidate[name])
for name in ('logLikelihood', 'nullLogLikelihood', 'likelihoodRatio'):
    assert math.isclose(independent[name], expected['fit'][name], rel_tol=0, abs_tol=tolerance.get(name, 1e-8)), (name, independent[name], expected['fit'][name])
{'status': 'PASS', 'command': fixture['command'], 'sets': independent['sets'], 'likelihoodRatio': independent['likelihoodRatio'], 'modelPValue': independent['modelPValue']}

In [ ]:
reversed_result = fit_conditional(list(reversed(records)))  # row-order invariance
relabeled = [{**record, 'GROUPID': 'SET-' + record['GROUPID']} for record in records]
relabeled_result = fit_conditional(relabeled)  # matched-set-label invariance
for result in (reversed_result, relabeled_result):
    assert np.allclose(result['coefficients'], independent['coefficients'], rtol=0, atol=1e-9)
    assert math.isclose(result['logLikelihood'], independent['logLikelihood'], rel_tol=0, abs_tol=1e-9)
{'row-order invariance': 'PASS', 'matched-set-label invariance': 'PASS'}

## Result and next gate

A clean run independently reproduces the browser candidate's conditional likelihood, coefficients, standard errors, adjusted odds ratios, Wald limits, and likelihood-ratio statistic for 100 one-case/two-control sets. This is candidate evidence—not legacy parity. Rust/WASM migration, categorical and interaction terms, exclusions/boundaries, and reviewed desktop Epi Info output remain required gates.